# Advanced 04 — Agent Attestations, Verifiable Credentials & Trust Evidence

**Scenario:** an enterprise claims agent requests a sensitive tool. The verifier requires independently issued evidence about registration, workload, release provenance, evaluation and governance before policy can authorize it.

> Educational code implements compact cryptographic patterns for learning. For production VC/SD-JWT/OpenID4VC protocols, use standards-conformant libraries and interoperability profiles.


In [ ]:
from datetime import datetime,timedelta,timezone
import base64,hashlib,json,copy,secrets
import pandas as pd
import networkx as nx
from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PrivateKey
NOW=datetime.now(timezone.utc)
def canonical(x): return json.dumps(x,sort_keys=True,separators=(",",":")).encode()
def b64(x): return base64.urlsafe_b64encode(x).decode().rstrip("=")
def ub64(x): return base64.urlsafe_b64decode(x+"="*(-len(x)%4))


## 1 — Claim vs evidence

In [ ]:
model_claim={"agent":"agent:claims","approved":True}
evidence_claim={"issuer":"did:web:registry.example","subject":"agent:claims","approved":True}
print("Model claim is untrusted:",model_claim)
print("Evidence becomes trusted only after verification:",evidence_claim)


## 2 — Create issuer keys

In [ ]:
registry_private=Ed25519PrivateKey.generate()
registry_public=registry_private.public_key()


## 3 — Create an agent registration credential

In [ ]:
vc={
"id":"urn:uuid:agent-reg-001",
"type":["VerifiableCredential","AgentRegistrationCredential"],
"issuer":"did:web:registry.example",
"validFrom":NOW.isoformat(),
"validUntil":(NOW+timedelta(days=30)).isoformat(),
"credentialSubject":{"id":"agent:claims","owner":"claims-platform","agentClass":"ClaimsAssistant"}
}
vc


## 4 — Sign the credential

In [ ]:
envelope={"credential":vc,"proof":{"type":"Ed25519Signature",
"value":b64(registry_private.sign(canonical(vc)))}}


## 5 — Verify proof

In [ ]:
registry_public.verify(ub64(envelope["proof"]["value"]),canonical(envelope["credential"]))
print("proof valid")


## 6 — Detect forgery

In [ ]:
forged=copy.deepcopy(envelope)
forged["credential"]["credentialSubject"]["agentClass"]="PaymentAdmin"
try:
    registry_public.verify(ub64(forged["proof"]["value"]),canonical(forged["credential"]))
    print("BAD")
except Exception:
    print("forgery detected")


## 7 — Trusted issuer policy

In [ ]:
trusted_issuers={
"did:web:registry.example":{"AgentRegistrationCredential"},
"did:web:evaluator.example":{"AgentEvaluationCredential"}
}
def issuer_allowed(c):
    return c["issuer"] in trusted_issuers and any(
        t in trusted_issuers[c["issuer"]] for t in c["type"])
issuer_allowed(vc)


## 8 — Valid signature from untrusted issuer

In [ ]:
attacker=Ed25519PrivateKey.generate()
evil=copy.deepcopy(vc);evil["issuer"]="did:web:attacker.example"
evil_sig=attacker.sign(canonical(evil))
print("Cryptographically signed:",True,"Locally trusted:",issuer_allowed(evil))


## 9 — Subject binding

In [ ]:
authenticated_agent="agent:claims"
assert vc["credentialSubject"]["id"]==authenticated_agent
substituted="agent:evil"
print("subject substitution accepted?",vc["credentialSubject"]["id"]==substituted)


## 10 — Freshness

In [ ]:
def temporal_valid(c,now):
    return datetime.fromisoformat(c["validFrom"]) <= now < datetime.fromisoformat(c["validUntil"])
temporal_valid(vc,NOW),temporal_valid(vc,NOW+timedelta(days=40))


## 11 — Status / revocation

In [ ]:
status={"urn:uuid:agent-reg-001":"active"}
def status_ok(c): return status.get(c["id"])=="active"
print(status_ok(vc))
status[vc["id"]]="revoked"
print(status_ok(vc))
status[vc["id"]]="active"


## 12 — Schema/profile semantics

In [ ]:
required={"id","type","issuer","validFrom","validUntil","credentialSubject"}
assert required.issubset(vc)
assert {"id","owner","agentClass"}.issubset(vc["credentialSubject"])


## 13 — Selective disclosure concept

In [ ]:
claims={"agent_id":"agent:claims","approved":True,"internal_risk":0.73,
"owner":"claims-platform","evaluation_details":"sensitive"}
requested={"agent_id","approved"}
presentation={k:v for k,v in claims.items() if k in requested}
presentation


## 14 — SD-JWT-style digest exercise

In [ ]:
# Simplified teaching model, NOT an SD-JWT implementation.
salt=secrets.token_hex(16)
disclosure=["internal_risk",0.73,salt]
digest=b64(hashlib.sha256(canonical(disclosure)).digest())
digest


## 15 — Verify a disclosure digest

In [ ]:
assert b64(hashlib.sha256(canonical(disclosure)).digest())==digest


## 16 — Unknown is not false

In [ ]:
verified={"approved":True}
def claim_state(name):
    if name not in verified:return "unknown"
    return "true" if verified[name] else "false"
claim_state("internal_risk")


## 17 — Model OpenID4VCI issuance

In [ ]:
oid4vci_flow=[
"discover credential issuer metadata",
"authorize/authenticate holder as required",
"obtain credential access token",
"send credential request",
"issuer returns credential",
"store outside LLM context"
]
pd.DataFrame({"step":range(1,len(oid4vci_flow)+1),"action":oid4vci_flow})


## 18 — Model OpenID4VP presentation

In [ ]:
presentation_request={
"client_id":"https://sensitive-tool.example",
"nonce":secrets.token_urlsafe(16),
"requested_evidence":["AgentRegistrationCredential","AgentEvaluationCredential"]
}
presentation_request


## 19 — Evidence wallet boundary

In [ ]:
llm_visible={"requested_action":"claim.update","claim_id":"483"}
runtime_only={"credential_handles":["agent-reg-001","eval-044"],"private_keys":"NOT_MODEL_VISIBLE"}
llm_visible,runtime_only


## 20 — Workload attestation

In [ ]:
workload={"spiffe_id":"spiffe://corp.example/prod/claims-agent",
"agent_id":"agent:claims","release_digest":"sha256:abc123","attested":True,
"observed_at":NOW.isoformat()}


## 21 — Bind workload to approved release

In [ ]:
approved_release={"agent_id":"agent:claims","digest":"sha256:abc123"}
assert workload["agent_id"]==approved_release["agent_id"]
assert workload["release_digest"]==approved_release["digest"]


## 22 — SLSA-style provenance

In [ ]:
provenance={
"_type":"https://in-toto.io/Statement/v1",
"subject":[{"name":"claims-agent","digest":{"sha256":"abc123"}}],
"predicateType":"https://slsa.dev/provenance/v1",
"predicate":{"buildDefinition":{"buildType":"https://example/build"},
"runDetails":{"builder":{"id":"https://ci.example/builder"}}}
}
provenance


## 23 — Verify artifact digest binding

In [ ]:
artifact_digest=provenance["subject"][0]["digest"]["sha256"]
assert workload["release_digest"]=="sha256:"+artifact_digest


## 24 — Release-bound evaluation

In [ ]:
evaluation={"issuer":"did:web:evaluator.example","agent_id":"agent:claims",
"release_digest":"sha256:abc123","suite":"agent-security-v7",
"passed":True,"valid_until":(NOW+timedelta(days=7)).isoformat()}
assert evaluation["release_digest"]==workload["release_digest"]


## 25 — Configuration fingerprint

In [ ]:
config={"model":"approved-model-v3","system_prompt_version":"p17",
"tools":["knowledge.search","claim.update"],"policy_bundle":"policy-v9"}
fingerprint=hashlib.sha256(canonical(config)).hexdigest()
fingerprint


## 26 — Governance credential

In [ ]:
governance={"issuer":"did:web:governance.example","agent_id":"agent:claims",
"release_digest":"sha256:abc123","approved":True,
"expires":(NOW+timedelta(days=14)).isoformat()}


## 27 — Agent card vs evidence

In [ ]:
agent_card={"purpose":"Assist claims adjusters","limitations":["No autonomous payment approval"],
"owner":"Claims Platform"}
print("Useful documentation:",agent_card)
print("Not sufficient cryptographic authorization evidence.")


## 28 — Trust mark

In [ ]:
trust_mark={"issuer":"enterprise-assurance-program","scheme":"agent-baseline-v2",
"subject":"agent:claims","valid":True}


## 29 — Build an assurance profile

In [ ]:
profile={
"identity":"high" if issuer_allowed(vc) and status_ok(vc) else "none",
"workload":"high" if workload["attested"] else "none",
"supply_chain":"high" if workload["release_digest"]=="sha256:"+artifact_digest else "none",
"evaluation":"high" if evaluation["passed"] else "low",
"governance":"high" if governance["approved"] else "none",
"runtime_risk":"low"
}
profile


## 30 — Evidence composition

In [ ]:
def sensitive_write(p):
    if p.get("quarantined"):return "deny"
    required=["identity","workload","supply_chain","evaluation","governance"]
    if all(p[x]=="high" for x in required) and p["runtime_risk"]=="low":
        return "allow"
    return "step_up"
sensitive_write(profile)


## 31 — Conflicting negative evidence

In [ ]:
conflicted={**profile,"quarantined":True}
sensitive_write(conflicted)


## 32 — Evidence graph

In [ ]:
g=nx.DiGraph()
g.add_edge("agent:claims","release:abc123",relation="approved_release")
g.add_edge("release:abc123","ci:builder",relation="built_by")
g.add_edge("release:abc123","evaluation:044",relation="evaluated_by")
g.add_edge("agent:claims","workload:current",relation="running_as")
g.add_edge("agent:claims","governance:approval",relation="approved_by")
list(g.edges(data=True))


## 33 — OPA-style verified facts

In [ ]:
opa_input={"evidence":{
"agent":{"registered":profile["identity"]=="high","quarantined":False},
"workload":{"attested":profile["workload"]=="high"},
"release":{"provenance_verified":profile["supply_chain"]=="high"},
"evaluation":{"passed":profile["evaluation"]=="high"},
"governance":{"approved":profile["governance"]=="high"}},
"risk":{"level":"low"}}
opa_input


## 34 — Cedar-style forbid precedence

In [ ]:
def cedar_decision(permit,forbid):
    if forbid:return "deny"
    return "allow" if permit else "deny"
cedar_decision(True,True),cedar_decision(True,False)


## 35 — Presentation replay challenge

In [ ]:
issued_nonce=presentation_request["nonce"]
used_nonces=set()
def accept_presentation(nonce):
    if nonce!=issued_nonce or nonce in used_nonces:return False
    used_nonces.add(nonce);return True
accept_presentation(issued_nonce),accept_presentation(issued_nonce)


## 36 — Assurance laundering

In [ ]:
evidence_subject="agent:claims"
caller="agent:research"
print("Can research agent reuse claims-agent evidence?",evidence_subject==caller)


## 37 — Issuer compromise

In [ ]:
issuer_state={"did:web:registry.example":"active"}
def issuer_operational(issuer):return issuer_state.get(issuer)=="active"
issuer_state["did:web:registry.example"]="compromised"
issuer_operational(vc["issuer"])


## 38 — Adversarial test matrix

In [ ]:
attacks=[
"forged signature","untrusted issuer","subject substitution","expired credential",
"revoked credential","workload/release mismatch","undisclosed claim misuse",
"assurance laundering","issuer compromise","conflicting quarantine",
"supply-chain digest mismatch","presentation replay"
]
pd.DataFrame({"attack":attacks,"expected":["deny"]*len(attacks)})


# Capstone — Enterprise Agent Trust Gate

Build a verifier for:

```text
Agent requests sensitive claim.update
          ↓
verify Agent Registration VC
          ↓
verify current workload attestation
          ↓
verify SLSA/in-toto release provenance
          ↓
verify release-bound security evaluation
          ↓
verify governance approval
          ↓
check revocation + freshness + negative evidence
          ↓
build assurance profile
          ↓
OPA/Cedar policy
          ↓
ALLOW / DENY / STEP-UP
```

Requirements:

1. model-generated claims never become trusted evidence;
2. only approved issuers can assert security-relevant credential types;
3. all proofs/signatures are verified;
4. subject identity matches the authenticated agent;
5. workload, release and evaluation digests agree;
6. expired/revoked evidence is rejected;
7. quarantine overrides positive evidence;
8. unknown selective-disclosure claims never grant privilege;
9. presentation replay is rejected;
10. sensitive evidence stays outside LLM context;
11. policy consumes normalized verified facts;
12. the decision records exactly which evidence was used.


# Review questions

1. What is the difference between a claim and an attestation?
2. What does a valid signature prove—and what does it not prove?
3. What are issuer, holder, subject and verifier?
4. Why is VC verification not authorization?
5. Why is subject binding critical for agents?
6. How should evidence freshness vary by claim type?
7. What problem does selective disclosure solve?
8. Why must `unknown` differ from `false`?
9. What roles do OpenID4VCI and OpenID4VP play?
10. Why should credentials/private keys remain outside model context?
11. How does workload attestation complement logical agent identity?
12. Why should evaluation evidence bind to a release digest?
13. How does SLSA/in-toto provenance contribute to agent assurance?
14. Why is an agent/system card not automatically enforcement evidence?
15. What is assurance laundering?
16. How should conflicting negative evidence be handled?
17. Why is a multidimensional assurance profile better than an opaque trust score?
18. Which evidence should be re-evaluated during a long-running agent session?
